### this code reads a czi file, opens in napari, lets you select an ROI and saves that roi-cropped region across both green and red channels across all z slices as a ome-tiff file with metadata

# 

 i have also added a max int proj to be able to aid the croppong

In [ ]:
#  Path Setup Cell
import os
imno = "15"  # only need to change here when switching images
ratio="1isto1" # ofc need to change this when changing 
date="05192025" # change among folders

base_path = f'/Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/{ratio}/{ratio}_{date}/'

input_czi_path = os.path.join(base_path, f'Image {imno}_{ratio}_{date}/Image {imno}_{ratio}_{date}.czi')
base_folder = os.path.join(base_path, f'Image {imno}_{ratio}_{date}')

colno = ''  # Default: no colony selected
outputpath = os.path.join(base_folder, f'tif file_Image {imno}_{ratio}_{date}')
filename = f"Image {imno}_{ratio}_{date}_pos01_t01_filtered_roi.ome.tiff"

if not os.path.exists(input_czi_path):
    raise FileNotFoundError(f"\n⚠️ File not found:\n{input_czi_path}\n\nDouble-check imno, ratio, or date.")
print(f"Base folder: {base_folder}\n")
print(f"Input path: {input_czi_path}\n")
print(f"Default output path: {outputpath}\n")

Base folder: /Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/1isto1/1isto1_05192025/Image 15_1isto1_05192025

Input path: /Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/1isto1/1isto1_05192025/Image 15_1isto1_05192025/Image 15_1isto1_05192025.czi

Default output path: /Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/1isto1/1isto1_05192025/Image 15_1isto1_05192025/tif file_Image 15_1isto1_05192025



In [ ]:
import os
import numpy as np
import napari
from aicsimageio import AICSImage
from aicsimageio.writers import OmeTiffWriter
from skimage.draw import polygon
import SimpleITK as sitk

# ------------- User Parameters -------------
czi_path = input_czi_path            # CZI file location.
selected_channels = [0, 2]           # Channels to extract. No need for DIC
apply_gaussian = True                # Set to True to apply Gaussian blurring.
gaussian_sigma = [0.0, 1.0, 1.0]     # Sigma for Z, Y, X respectively.
                                     # (No filtering in Z; filtering in Y and X.)

# ------------- Step 1: Load the Image & Extract Metadata -------------
print("Loading image...")
img = AICSImage(czi_path)
data = img.data
print("Full image data shape:", data.shape)
print("Dimension info:", img.dims)

# The image shape is (T, C, Z, Y, X) with T=1. Remove the time dimension.
data = data[0]  # Now shape becomes: (C, Z, Y, X)
print("Data shape after removing time dimension:", data.shape)

# Extract the selected channels.
data_sub = data[np.array(selected_channels), ...]
print("Data for selected channels shape:", data_sub.shape)

# ------------- Retrieve Physical Voxel Sizes -------------
voxel_sizes = img.physical_pixel_sizes
x_scale = voxel_sizes.X if voxel_sizes.X is not None else 1.0
y_scale = voxel_sizes.Y if voxel_sizes.Y is not None else 1.0
z_scale = voxel_sizes.Z if voxel_sizes.Z is not None else 1.0
print(f"Voxel sizes (µm): X={x_scale}, Y={y_scale}, Z={z_scale}")

# ------------- Step 2: Set Up Napari Viewer for Interactive ROI -------------
viewer = napari.Viewer(ndisplay=2)
print("Current working directory:", os.getcwd())

# Add the selected channels image (assumed shape: (C, Z, Y, X)).
viewer.add_image(data_sub, name='Selected Channels', channel_axis=0)

# Determine image dimensions (Y, X) from data_sub.
img_height = data_sub.shape[2]
img_width = data_sub.shape[3]

# Define the border coordinates in (row, col) = (y, x) format.
border_coords = np.array([
    [0, 0],
    [0, img_width - 1],
    [img_height - 1, img_width - 1],
    [img_height - 1, 0]
])

# Add a shapes layer that draws the border outline. #w/o this i would end up crashing napari because the the ROI outside image boundries will get NaNs or something ; stay inside boundries
viewer.add_shapes(
    data=[border_coords],
    name='Image Border Outline',
    shape_type='polygon',
    edge_color='red',
    face_color='transparent',  # no fill
    edge_width=3
)
print("Added image border outline to the viewer.")

# --- NEW: Compute & add maximum projection of channel2 as a guide ---
channel2 = data_sub[1]  # shape: (Z, Y, X)
max_proj_ch2 = np.max(channel2, axis=0)
viewer.add_image(
    max_proj_ch2,
    name='Channel2 Max Projection',
    blending='additive',
    colormap='gray',
    opacity=0.7
)
print("Added max projection of channel2 to the viewer.")

# --- NEW: Compute & add maximum projection of channel0 as a guide ---
channel0 = data_sub[0]  # shape: (Z, Y, X)
max_proj_ch0 = np.max(channel0, axis=0)
viewer.add_image(
    max_proj_ch0,
    name='Channel0 Max Projection',
    blending='additive',
    colormap='magma',
    opacity=0.7
)
print("Added max projection of channel0 to the viewer.")

# Add a shapes layer for polygon ROI drawing.
shapes_layer = viewer.add_shapes(name='ROI', shape_type='polygon')
print("Draw a polygon ROI on the 'ROI' layer and then press 'w'.")

# ------------- Step 3: Define the ROI Processing, Filtering, & Saving Callback -------------
def save_roi(event):
    print("Save ROI callback triggered.")
    if len(shapes_layer.data) == 0:
        print("No ROI polygon drawn! Please draw a polygon ROI before pressing 'w'.")
        return

    roi_coords = shapes_layer.data[0]
    print("ROI polygon vertices:", roi_coords)

    min_y, min_x = np.floor(np.min(roi_coords, axis=0)).astype(int)
    max_y, max_x = np.ceil(np.max(roi_coords, axis=0)).astype(int)
    print(f"ROI bounding box: x from {min_x} to {max_x}, y from {min_y} to {max_y}")

    roi_height = max_y - min_y
    roi_width = max_x - min_x

    shifted_coords = roi_coords - np.array([min_y, min_x])
    rr, cc = polygon(shifted_coords[:, 0], shifted_coords[:, 1], shape=(roi_height, roi_width))
    mask = np.zeros((roi_height, roi_width), dtype=bool)
    mask[rr, cc] = True
    print("Polygon mask created with shape:", mask.shape)

    cropped_data = data_sub[:, :, min_y:max_y, min_x:max_x].copy()
    print("Cropped bounding box shape:", cropped_data.shape)

    for ch in range(cropped_data.shape[0]):
        for z in range(cropped_data.shape[1]):
            slice_img = cropped_data[ch, z, :, :]
            slice_img[~mask] = 0
            cropped_data[ch, z, :, :] = slice_img

    if apply_gaussian:
        filtered_channels = []
        for ch in range(cropped_data.shape[0]):
            channel_array = cropped_data[ch]
            sitk_image = sitk.GetImageFromArray(channel_array)
            gaussian_filtered = sitk.DiscreteGaussian(sitk_image, gaussian_sigma)
            filtered_array = sitk.GetArrayFromImage(gaussian_filtered)
            filtered_channels.append(filtered_array)
        final_data = np.stack(filtered_channels, axis=0)
        print("Applied Gaussian filtering to the ROI.")
    else:
        final_data = cropped_data

    # ----------- NEW CODE TO... save to outputpath -----------

    base_name = os.path.splitext(os.path.basename(czi_path))[0]

    # Determine colony suffix
    colony_suffix = f"_colony{colno}" if colno else "" # have to repeat with the same image if the image has multiple aggregate ; just change the colno varialbe 
    
    if apply_gaussian:
        filename = f"{base_name}{colony_suffix}_pos01_t01_filtered_roi.ome.tiff"
    else:
        filename = f"{base_name}{colony_suffix}_pos01_t01_roi.ome.tiff"
    
    full_output_path = os.path.join(outputpath, filename)


    try:
        OmeTiffWriter.save(
            final_data,
            full_output_path,
            dim_order="CZYX",
            physical_pixel_sizes=voxel_sizes
        )
        print(f"OME-TIFF saved with voxel size metadata: {full_output_path}")
    except Exception as e:
        print("Error during OME-TIFF save:", e)

@viewer.bind_key('w')
def key_save(event):
    print("'w' key pressed.")
    save_roi(event)

# Run the napari event loop.
napari.run()

Loading image...
Full image data shape: (1, 3, 72, 1024, 1024)
Dimension info: <Dimensions [T: 1, C: 3, Z: 72, Y: 1024, X: 1024]>
Data shape after removing time dimension: (3, 72, 1024, 1024)
Data for selected channels shape: (2, 72, 1024, 1024)
Voxel sizes (µm): X=0.1647352852086885, Y=0.1647352852086885, Z=0.6
Current working directory: /Users/jgosai/Library/CloudStorage/GoogleDrive-jgosai@asu.edu/My Drive/confocal mixed aggregates imaging/2isto1/2isto1_05202025
Added image border outline to the viewer.
Added max projection of channel2 to the viewer.
Added max projection of channel0 to the viewer.
Draw a polygon ROI on the 'ROI' layer and then press 'w'.
'w' key pressed.
Save ROI callback triggered.
ROI polygon vertices: [[  62.377193   198.91052  ]
 [  12.0754385  312.6883   ]
 [  49.202923   397.72223  ]
 [ 119.864914   428.8614   ]
 [ 379.75732    479.16315  ]
 [ 717.4977     529.4649   ]
 [1009.7269     471.9772   ]
 [1010.92456    364.1877   ]
 [ 955.83215    255.20059  ]
 [ 929